# SecureRAG Tutorial 1: Framework Basics

This notebook walks through a minimal end-to-end SecureRAG flow:

1. Configure a protocol and backend
2. Build a corpus
3. Run the SecureRAG agent
4. Inspect answer and budget

In [1]:
from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.corpus import CorpusBuilder
from securerag.llm import ModelAgentLLM
from securerag.models import RawDocument

In [2]:
docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Security policy requires quarterly risk treatment tracking and owner assignment."),
    RawDocument(doc_id="ops", text="Operational incidents increased in July due to queue saturation in ingestion service."),
]

config = PrivacyConfig(
    protocol=PrivacyProtocol.ENCRYPTED_SEARCH,
    backend="rust://local",
    top_k=3,
    max_rounds=4,
    encrypted_search_scheme="structured",
    structured_use_bigrams=True,
)

In [3]:
corpus = (
    CorpusBuilder(config.protocol, backend_url=config.backend)
    .with_encrypted_search_scheme(
        config.encrypted_search_scheme,
        structured_use_bigrams=config.structured_use_bigrams,
    )
    .with_chunk_size(120)
    .add_documents(docs)
    .build()
)

print("Corpus type:", type(corpus).__name__)
print("Index size:", corpus.index_size())

Corpus type: SSECorpus
Index size: 3


In [4]:
llm = ModelAgentLLM(
    provider="ollama",
    model="qwen3:0.6b",
    use_ollama=False,
    use_huggingface=False,
)

agent = SecureRAGAgent.from_config(config, corpus=corpus, llm=llm)
result = agent.run("Summarize key risks in Q3")

print(result.answer)
print(agent.budget_snapshot())

Answer for: Summarize key risks in Q3

Evidence:
- [q3] Q3 risk report highlights vendor concentration and delayed remediation. (score=0.083)
- [policy] Security policy requires quarterly risk treatment tracking and owner assignment. (score=0.037)
- [ops] Operational incidents increased in July due to queue saturation in ingestion service. (score=0.033)
{'spent': 0.0, 'remaining': 1.0, 'rounds': 0, 'ledger': []}
